<div align='center'> <p> <b> WUM 2026, projekt 2  <p> </div>

<div align='center'> <b>Autor</b>: Julia Stasiak</div>
<div align='center'> <b>Data</b>: 29.05.2026</div>


In [1]:
# --- Instalacja ---
%pip install pandas scikit-learn

# --- Importy ---
import warnings

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.linear_model import (
    ElasticNet,
    LinearRegression,
    LogisticRegression,
    Lasso,
    Ridge
)

from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

warnings.filterwarnings('ignore', category=FutureWarning)

- - -

## 1. Wczytanie danych

In [ ]:
df = pd.read_csv('zad2_wum_data_for_students.csv', sep=';')
df

,Class,Output,Input1,Input2,Input3,Input4,Input5,Input6,Input7,Input8,...,Input391,Input392,Input393,Input394,Input395,Input396,Input397,Input398,Input399,Input400
0,0,0.800586,-0.002583,2.184037,-0.322008,1.621241,1.192444,-0.278356,-0.207366,0.735689,...,-2.140861,1.187660,0.345238,-0.844885,0.580007,-2.605781,-0.299471,0.711487,-0.753316,0.728763
1,0,2.168475,0.668637,1.373933,-0.476868,-0.724704,0.031162,-1.845921,0.784890,1.508526,...,-1.286120,-0.900044,-0.500399,-0.126421,-0.632233,-2.557419,0.056044,0.634774,-0.259835,0.106390
2,1,-1.210777,-0.681438,-0.544753,0.441346,-0.019906,-0.192135,-0.162510,-0.998777,0.686472,...,-0.391605,-0.190147,0.793746,-0.812737,-0.068228,-0.313143,2.564096,0.848355,0.180556,-1.525615
3,1,0.505678,-0.497957,0.720712,0.149120,0.019251,1.377850,0.981337,-0.846813,0.036790,...,-0.176734,-0.947351,-0.888601,1.509450,-0.501929,-0.554909,-0.104051,0.442150,-0.056644,1.447267
4,1,-10.281033,-1.178544,0.176941,1.112202,1.234189,0.999451,-0.773329,-0.811075,1.550537,...,-0.181325,0.198960,-0.697497,-0.836371,1.652071,0.974292,1.584071,-0.202352,1.362426,1.023857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1,-4.298039,-0.893128,2.081556,0.796121,0.436108,-0.849635,1.129482,1.432813,-0.438694,...,-0.474294,0.039143,1.808243,-0.034847,-1.314878,-1.235939,1.010456,-2.186403,-0.157829,0.738539
1996,1,-0.431692,0.048336,0.770285,-0.354350,-1.557706,-0.182954,-0.665730,0.322526,0.658221,...,1.087032,0.095401,0.301200,1.776995,-2.045261,-1.931008,-0.683551,0.000835,-0.671151,-0.945843
1997,1,-0.056681,1.404051,-0.061405,0.180448,-0.362992,0.826353,-0.066654,0.987946,-1.266302,...,1.011431,0.458901,-0.220498,0.004950,-1.928972,-1.574129,1.421012,-0.736559,-0.540174,-1.182067
1998,0,-0.983396,-0.474238,-1.288631,-0.326170,-0.275383,-0.315331,-1.225622,-0.656750,0.777151,...,-1.146012,-0.031477,-2.461869,1.037240,0.366076,-0.541171,-0.126733,-0.265069,-0.080381,0.166985


# **Task 1**: Baseline models

Tworzymy 2 proste modele w celu ustalenia punktów odniesienia określając wartości startowe $R^2$ i accuracy.

Wykorzystamy do tego:
* **regresje liniową** dla zmiennej *"Output"*,
* **regresje logistyczną** dla zmiennej *"Class"*,


Modele trenujemy na wszystkich (400) dostępnych cechach, bez dodatkowego przetwarzania. Aby ocenić ich skuteczność, dzielimy dane na zbiór treningowy (80%) i testowy (20%) oraz dodatkowo weryfikujemy, przeprowadzając 10-krotną walidację krzyżową.

In [ ]:
X = df.drop(['Class', 'Output'], axis=1)
y_outcome = df['Output']
y_class = df['Class']

# A: bazowy model regresji liniowej
X_lin_train, X_lin_test, y_lin_train, y_lin_test = train_test_split(X, y_outcome, test_size=0.2, random_state=55)

model_linear = LinearRegression()
model_linear.fit(X_lin_train, y_lin_train)

y_lin_train_predict = model_linear.predict(X_lin_train)
y_lin_test_predict = model_linear.predict(X_lin_test)

r2_train = r2_score(y_lin_train, y_lin_train_predict)
r2_test = r2_score(y_lin_test, y_lin_test_predict)

cv = KFold(n_splits=10, random_state=55, shuffle=True)
cv_lin_scores = cross_val_score(model_linear, X, y_outcome, cv=cv, scoring="r2")

print("---------------- BAZOWY MODEL REGRESJI LINIOWEJ ----------------")
print(f"R^2 na danych treningowych: {r2_train:.4f}")
print(f"R^2 na danych testowych: {r2_test:.4f}")
print(f"Średnie CV R^2: {cv_lin_scores.mean():.4f}")
print("----------------------------------------------------------------")

# B: bazowy model regresji logistycznej
X_log_train, X_log_test, y_log_train, y_log_test = train_test_split(X, y_class, test_size=0.2, random_state=55)

model_logistic = LogisticRegression()
model_logistic.fit(X_log_train, y_log_train)

y_log_train_predict = model_logistic.predict(X_log_train)
y_log_test_predict = model_logistic.predict(X_log_test)

accuracy_train = accuracy_score(y_log_train, y_log_train_predict)
accuracy_test = accuracy_score(y_log_test, y_log_test_predict)

cv = StratifiedKFold(n_splits=10, random_state=55, shuffle=True)
cv_log_scores = cross_val_score(model_logistic, X, y_class, cv=cv, scoring="accuracy")

print("---------------- BAZOWY MODEL REGRESJI LOGISTYCZNEJ ----------------")
print(f"Accuracy na danych treningowych: {accuracy_train:.4f}")
print(f"Accuracy na danych testowych: {accuracy_test:.4f}")
print(f"Średnie CV accuracy: {cv_log_scores.mean():.4f}")
print("---------------------------------------------------------------------")

---------------- BAZOWY MODEL REGRESJI LINIOWEJ ----------------
R^2 na danych treningowych: 0.6465
R^2 na danych testowych: 0.2422
Średnie CV R^2: 0.3747
----------------------------------------------------------------
---------------- BAZOWY MODEL REGRESJI LOGISTYCZNEJ ----------------
Accuracy na danych treningowych: 0.7306
Accuracy na danych testowych: 0.5200
Średnie CV accuracy: 0.5125
---------------------------------------------------------------------


### Komentarz do wyników:
Po wytrenowaniu modeli bazowych uzyskaliśmy mało zadowalające rezultaty:
#### Regresja liniowa:
* **$R^2$ na zbiorze treningowym**: ~0.65
* **$R^2$ na zbiorze testowym**: ~0.24
* **$R^2$ walidacja krzyżowa (średnia z 10-krotnej)**: ~0.37

**Wnioski**: Doszło do overfittingu. Model dobrze poradził sobie na zbiorze treningowym, ale ponad dwukrotny spadek w przypadku zbioru testowego sugeruje słabą zdolność do generalizacji bazowego modelu regresji liniowej. Słaby średni wynik z 10-krotnej walidacji krzyżowej tylko potwierdza niestabilność modelu. Takie wyniki sugerują, że w naszych danych jest wiele szumu i niepotrzebnych informacji.

##### Regresja logistyczna:
* **Accuracy na zbiorze treningowym**: ~0.73
* **Accuracy na zbiorze testowym**: ~0.52
* **Accuracy walidacja krzyżowa (średnia z 10-krotnej)**: ~0.51

**Wnioski**: Model osiągnął znacznie lepszą skuteczność niż regresja liniowa, jednak znowu spadek na danych testowych i walidacji krzyżowej do ~0.5 dla zmiennej binarnej "Class" wskazuje na niezadowalające nas działanie modelu, taki wynik dla danych binarnych równie dobrze można osiągnąć zgadując. Znowu wskazuje to na duży szum i nadmiar nieistotnych informacji.


- - -

# **Task 2**: More advanced classification

Model bazowy ukazał, że dane posiadają duży szum informacyjny, aby sobie z tym poradzić stworzymy nowe, bardziej zaawansowane modele, rozszerzając je również o *selekcje cech* oraz *optymalizację parametrów*.

* **selekcja cech**: zauważyliśmy, że klasa zależy tylko od niektórych predykatorów, zatem dla wybranych modeli (Random Forest oraz SVC) użyjemy *SelectKBest*, aby odrzucić nieistotne wartości przed treningiem modelu.
* **optymalizacja parametrów**: używamy *GridSearchCV* z 10-krotną walidacją krzyżową, aby znależć optymalne parametry, których później użyjemy do wytrenowania modelu.


**Bardziej zaawansowane modele**:
* **kNN**: sprawdzamy, czy klasy tworzą skupiska,
* **Random Forest**: używamy do wykrycia ewentualnej, skomplikowanej nieliniowości,
* **SVC z jądrem RBF**: używamy go, aby poradzić sobie z nieliniowymi i złożonymi, nakładającymi się na siebie danymi.

In [ ]:
cv = StratifiedKFold(n_splits=10, random_state=55, shuffle=True)

# Model kNN
model_knn = KNeighborsClassifier(n_neighbors=55)
model_knn.fit(X_log_train, y_log_train)

y_knn_train_predict = model_knn.predict(X_log_train)
y_knn_test_predict = model_knn.predict(X_log_test)

accuracy_train = accuracy_score(y_log_train, y_knn_train_predict)
accuracy_test = accuracy_score(y_log_test, y_knn_test_predict)
cv_knn_scores = cross_val_score(model_knn, X, y_class, cv=10, scoring="accuracy")

print("---------------- kNN ----------------")
print(f"Accuracy na danych testowych: {accuracy_test:.4f}")
print(f"Średnie CV accuracy: {cv_knn_scores.mean():.4f}")
print("-------------------------------------")

# Model RandomForest z GridSearchCV
pipeline_random_forest = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('random_forest', RandomForestClassifier(random_state=55))
])

param_grid = {
    'feature_selection__k': [3, 5, 10, 15, 20, 'all'],
    'random_forest__n_estimators': [100, 200],
    'random_forest__max_depth': [3, 4, 5, 30],
    'random_forest__min_samples_split': [10, 20, 30],
    'random_forest__min_samples_leaf': [10, 15, 20]
}

grid_search = GridSearchCV(estimator=pipeline_random_forest, param_grid=param_grid,cv=cv, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_log_train, y_log_train)

best_random_forest = grid_search.best_estimator_

y_random_forest_train_predict = best_random_forest.predict(X_log_train)
y_random_forest_test_predict = best_random_forest.predict(X_log_test)

print("---------------- RANDOM FOREST PIPELINE ----------------")
print(f"Najlepsze znalezione parametry {grid_search.best_params_}")
print(f"Accuracy na danych testowych: {accuracy_score(y_log_test, y_random_forest_test_predict):.4f}")
print(f"Średnie CV accuracy: {grid_search.best_score_:.4f}")
print("------------------------------------------------")

# Model SVC z GridSearchCV
pipeline_svc = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('svc', SVC(random_state=55))
])

param_grid = {
    'feature_selection__k': [10, 12, 14, 'all'],
    'svc__C': [25, 50, 80, 100],
    'svc__kernel': ['rbf'],
    'svc__gamma': ['scale', 'auto', 0.05, 0.1, 0.2]
}

grid_search = GridSearchCV(estimator=pipeline_svc, param_grid=param_grid, cv=cv, scoring="accuracy", n_jobs=-1)

grid_search.fit(X_log_train, y_log_train)
best_pipeline_svc = grid_search.best_estimator_

y_svc_train_predict = best_pipeline_svc.predict(X_log_train)
y_svc_test_predict = best_pipeline_svc.predict(X_log_test)

print("---------------- SVC PIPELINE ----------------")
print(f"Najlepsze znalezione parametry {grid_search.best_params_}")
print(f"Accuracy na danych testowych: {accuracy_score(y_log_test, y_svc_test_predict):.4f}")
print(f"Średnie CV accuracy: {grid_search.best_score_:.4f}")
print("----------------------------------------------")

feature_selector = best_pipeline_svc.named_steps['feature_selection']
feature_selected = [feature for feature, use in zip(X_log_train, feature_selector.get_support()) if use]
print("Wybrane cechy:")
for feature in feature_selected:
    print(feature)

---------------- kNN ----------------
Accuracy na danych testowych: 0.6000
Średnie CV accuracy: 0.5575
-------------------------------------
---------------- RANDOM FOREST PIPELINE ----------------
Najlepsze znalezione parametry {'feature_selection__k': 10, 'random_forest__max_depth': 30, 'random_forest__min_samples_leaf': 10, 'random_forest__min_samples_split': 10, 'random_forest__n_estimators': 200}
Accuracy na danych testowych: 0.7350
Średnie CV accuracy: 0.7600
------------------------------------------------
---------------- SVC PIPELINE ----------------
Najlepsze znalezione parametry {'feature_selection__k': 10, 'svc__C': 50, 'svc__gamma': 0.2, 'svc__kernel': 'rbf'}
Accuracy na danych testowych: 0.8350
Średnie CV accuracy: 0.8031
----------------------------------------------
Wybrane cechy:
Input2
Input40
Input41
Input74
Input238
Input240
Input293
Input308
Input330
Input396


### Komentarz do wyników:
Zastosowanie bardziej zaawansowanych modeli oraz użycie selekcji cech i optymalizacji parametrów przyniosło widoczną poprawę względem modelu bazowego.

#### Porównanie modeli:
* **kNN**: najmniejsze accuracy ze wszystkich 3 modeli na poziomie ~0.6, większe jednak niż modelu bazowego. Bez dodatkowej selekcji cech wciąż podlegał dużemu zaszumieniu.
* **Random Forest**, z selekcją cech (k=10) i optymalizacją parametrów (max_depth=30, n_estimators=200) osiąga już znacznie lepsze accuracy na poziomie ~0.78 na danych testowych. Redukcja szumów przyniosła efekty.
* **SVC**, z selekcją cech (k=10) i optymalizacją parametrów (C=50, gamma=0.2, kernel=rbf) zdecydowanie osiąga najlepsze accuracy na poziomie > 0.8 (~0.83 na testowym i 0.81 CV).

Zarówno dla Random Forest jak i SVC wybrano tyle samo, takich samych cech:
- Input2
- Input40
- Input41
- Input74
- Input238
- Input240
- Input293
- Input308
- Input330
- Input396

*Wnioski*: Najlepszy z modeli (**SVC**) poprawił zdolność do generalizacji z ~0.52 dla bazowego modelu do ~0.83. Zarówno dla Random Forest jak i SVC, GridSearchCV wybrał k=10 jako optymalne, co sugeruje, że z pierwotnych danych znaczna większość była nieistotnym szumem psującym zdolność do generalizacji.

- - -

# **Task 3**: More advanced Regression
Podobnie jak model bazowy dla regresji logistycznej, ten dla regresji liniowej również cechował się dużym overfittingiem.

Zastosujemy zaawansowane modele regularyzacji, które nakładają kary na model:
* **Ridge** z karą L2: zmniejsza wagi mniej ważnych cech, ale ich nie usuwa,
* **Lasso** z karą L1: skutecznie dokonuje selekcji cech przez ignorowanie mniej istotnych danych,
* **ElasticNet** łączy zalety kary L1 i L2, pozwalając na jednoczesne zmniejszanie wag oraz eliminację niepotrzebnych cech.


Podobnie jak w podpunkcie 2 urzyjemy 'Pipeline' oraz za pomocą GridSearchCV z 10-krotną walidacją krzyżową wybierzemy optymalne parametry.

In [ ]:
cv = KFold(n_splits=10, random_state=55, shuffle=True)

# Model Ridge z GridSearchCV
pipeline_ridge = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_regression)),
    ('ridge', Ridge(random_state=55, max_iter=50000)),
])

param_grid_ridge = {
    'feature_selection__k': [10, 20, 30, 40, 'all'],
    'ridge__alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.1],
}

grid_search = GridSearchCV(estimator=pipeline_ridge, param_grid=param_grid_ridge, cv=cv, scoring="r2", n_jobs=-1)
grid_search.fit(X_lin_train, y_lin_train)
best_pipeline_ridge = grid_search.best_estimator_

y_ridge_test_predict = best_pipeline_ridge.predict(X_lin_test)

print("---------------- RIDGE PIPELINE ----------------")
print(f"Najlepsze znalezione parametry {grid_search.best_params_}")
print(f"R^2 na danych testowych: {r2_score(y_lin_test, y_ridge_test_predict):.4f}")
print(f"Średnie CV accuracy: {grid_search.best_score_:.4f}")
print("-------------------------------------------------")

# Model ElasticNet z GridSearchCV
pipeline_elasticNet = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_regression)),
    ('elasticnet', ElasticNet(random_state=55, max_iter=50000)),
])

param_grid_elastic = {
    'feature_selection__k': [10, 20, 30, 40, 'all'],
    'elasticnet__alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.1],
    'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.6, 0.7]
}

grid_search = GridSearchCV(estimator=pipeline_elasticNet, param_grid=param_grid_elastic, cv=cv, scoring="r2", n_jobs=-1)
grid_search.fit(X_lin_train, y_lin_train)
best_pipeline_elastic = grid_search.best_estimator_

y_ekasticnet_test_predict = best_pipeline_elastic.predict(X_lin_test)

print("---------------- ELASTICNET PIPELINE ----------------")
print(f"Najlepsze znalezione parametry {grid_search.best_params_}")
print(f"R^2 na danych testowych: {r2_score(y_lin_test, y_ekasticnet_test_predict):.4f}")
print(f"Średnie CV accuracy: {grid_search.best_score_:.4f}")
print("-------------------------------------------------------")

# Model Lasso z GridSearchCV
pipeline_lasso = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_regression)),
    ('lasso', Lasso(random_state=55, max_iter=50000)),
])

param_grid = {
    'feature_selection__k': [10, 20, 30, 40, 'all'],
    'lasso__alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
}

grid_search = GridSearchCV(estimator=pipeline_lasso, param_grid=param_grid, cv=cv, scoring="r2", n_jobs=-1)
grid_search.fit(X_lin_train, y_lin_train)
best_pipeline_lasso = grid_search.best_estimator_

y_lasso_test_predict = best_pipeline_lasso.predict(X_lin_test)

print("---------------- LASSO PIPELINE ----------------")
print(f"Najlepsze znalezione parametry {grid_search.best_params_}")
print(f"R^2 na danych testowych: {r2_score(y_lin_test, y_lasso_test_predict):.4f}")
print(f"Średnie CV accuracy: {grid_search.best_score_:.4f}")
print("-------------------------------------------------")

feature_selector_lasso = best_pipeline_lasso.named_steps['feature_selection']
feature_selected_lasso = [feature for feature, use in zip(X_log_train, feature_selector_lasso.get_support()) if use]
lasso_model = best_pipeline_lasso.named_steps['lasso']
coefficients = lasso_model.coef_
feature_importance = list(zip(feature_selected_lasso, coefficients))
feature_importance_sorted = sorted(feature_importance, key=lambda x: x[1], reverse=True)
print("---------------- Najważniejsze 10 cech: ----------------")
for feature, coef in feature_importance_sorted[:10]:
    print(f"{feature}: {coef:.4f}")
zeroed_features = [feature for feature, coef in feature_importance_sorted if coef == 0.0]
print(f"Najmniej ważne cechy - liczba wyzerowanych przez Lasso: {len(zeroed_features)}")

---------------- RIDGE PIPELINE ----------------
Najlepsze znalezione parametry {'feature_selection__k': 30, 'ridge__alpha': 0.1}
R^2 na danych testowych: 0.4214
Średnie CV accuracy: 0.4735
-------------------------------------------------
---------------- ELASTICNET PIPELINE ----------------
Najlepsze znalezione parametry {'elasticnet__alpha': 0.1, 'elasticnet__l1_ratio': 0.7, 'feature_selection__k': 'all'}
R^2 na danych testowych: 0.4344
Średnie CV accuracy: 0.4891
-------------------------------------------------------
---------------- LASSO PIPELINE ----------------
Najlepsze znalezione parametry {'feature_selection__k': 'all', 'lasso__alpha': 0.1}
R^2 na danych testowych: 0.4412
Średnie CV accuracy: 0.4980
-------------------------------------------------
---------------- Najważniejsze 10 cech: ----------------
Input83: 0.8832
Input223: 0.8794
Input167: 0.7308
Input193: 0.7297
Input342: 0.6809
Input292: 0.6702
Input184: 0.5403
Input136: 0.4997
Input173: 0.4430
Input18: 0.4069
Najm

### Komentarz:
Modele osiągnęły podobne wartości $R^2$ :
* **Ridge**: ~0.42 testowy, ~0.47 CV
* **ElasticNet**: ~0.43 testowy, ~0.49 CV
* **Lasso**: ~0.44 testowy ~0.50 CV

Najlepszy, zoptymalizowany model Lasso osiągnął najlepsze wyniki dla 10-krotnej walidacji krzyżowej osiągając $R^2$ = \~0.5, oznacza to znaczną poprawę względem modelu bazowego (~0.37 dla CV).

Wybrany, najlepszy model Lasso wybrał wszystkie cechy do użycia, jednak:
* Spośród 400 cech, model na skutek nałożenia kary L1 aż 350 wyzerował.
* Algorytm wyznaczył 10 cech, które miały największy wpływ na wynik:
  * Input83: 0.8832
  * Input223: 0.8794
  * Input167: 0.7308
  * Input193: 0.7297
  * Input342: 0.6809
  * Input292: 0.6702
  * Input184: 0.5403
  * Input136: 0.4997
  * Input173: 0.4430
  * Input18: 0.4069

- - -

### Funkcja przetwarzająca "validation_data.csv" z wykorzystaniem 4 modeli:
* Bazowa regresja liniowa,
* Bazowa regresja logistyczna,
* Najlepszy model dla regresji: Lasso (k=all - 400, alpha=0.1),
* Najlepszy model dla klasyfikacji: SVC (k=10, C=50, gamma=0.2, kernel=rbf).

In [ ]:
def process(file, base_reg_model, base_class_model, best_reg_model, best_class_model):
    df = pd.read_csv(file, sep=';')

    X = df.drop(['Class', 'Output'], axis=1)
    y_outcome = df['Output']
    y_class = df['Class']

    r2_base_model = r2_score(y_outcome, base_reg_model.predict(X))
    accuracy_base_model = accuracy_score(y_class, base_class_model.predict(X))

    r2_best_model = r2_score(y_outcome, best_reg_model.predict(X))
    accuracy_best_model = accuracy_score(y_class, best_class_model.predict(X))

    print("---------------- WYNIKI ----------------")
    print(f"Model bazowy R^2: {r2_base_model}")
    print(f"Model najlepszy R^2: {r2_best_model}")
    print(f"Model bazowy accuracy: {accuracy_base_model}")
    print(f"Model najlepszy accuracy: {accuracy_best_model}")
    print("-----------------------------------------")

process('validation_data.csv', model_linear, model_logistic, best_pipeline_lasso, best_pipeline_svc)